Viewed deferred-init.ipynb:1-102
Viewed 3.3-deferred-init.ipynb:1-152

[deferred-init.ipynb](file:///Users/songchao/WorkSpace/PythonProject/DeepLearn/pytorch/chapter_deep-learning-computation/deferred-init.ipynb) 属于《动手学深度学习》（D2L）**深度学习计算**章节，主要讨论了深度学习框架中的**延后初始化（Deferred Initialization / Lazy Initialization）**机制。

以下是该 Notebook 的核心内容梳理与深度解析：

---

### 一、 核心概念：什么是延后初始化？

在传统的神经网络定义中，通常需要明确写出每一层的**输入维度**和**输出维度**。例如全连接层：
`y = x · W + b`
其中 `W` 的形状为 `(in_features, out_features)`。如果不知道输入特征数，框架就无法在内存中分配权重矩阵。

* **延后初始化的定义**：定义网络架构时，**仅指定输出维度，不指定输入维度**。框架此时并不立即在内存中开辟空间初始化权重参数；直到**第一批真实数据传入模型（First Forward Pass）**时，框架才根据输入数据的实际形状（如 `X.shape`）动态推断参数大小并完成参数初始化。
* **为什么需要延后初始化**：
  1. **降低心智负担**：无需手动计算复杂网络（特别是卷积神经网络中卷积、池化后多维展平）各层间的维度衔接。
  2. **提高代码灵活性**：修改前端网络或输入分辨率时，后续层无需手动重新计算输入通道数或特征维数。

---

### 二、 工作机制与推断流程

1. **实例化阶段（Instantiation）**：
   * 仅构建层之间的连接图，权重参数处于“未确定形状 / 未初始化”的占位状态。
   * 此时访问权重参数通常会提示形状未定义（未分配内存）。

2. **首次前向传播（First Forward Pass / Warm-up）**：
   * 将形状为 `(batch_size, d_in)` 的输入张量 `X` 送入网络。
   * **推断第一层**：框架根据输入维数 `d_in` 与该层目标输出维数 `d_out1`，确定第一层权重矩阵形状为 `(d_in, d_out1)`，并调用初始化算法（如高斯分布、Xavier 等）分配内存并初始化数值。
   * **级联推断后续层**：第一层的输出维度 `d_out1` 自动作为第二层的输入维度，以此类推，完成整条计算图上所有层的参数形状推导和参数分配。
   * 一旦参数形状被固定，后续再有批次数据流入时，将直接复用已初始化的参数，不再重复推断。

---

### 三、 PyTorch 中的延后初始化实践

在 D2L 原书中，早期 MXNet/Gluon 天然支持延迟初始化，而 PyTorch 传统设计倡导显式指定（如 `nn.Linear(in_features, out_features)`）。在现代 PyTorch 中，已提供了原生及灵活的替代方案（正如你在 [3.3-deferred-init.ipynb](file:///Users/songchao/WorkSpace/PythonProject/DeepLearn/mypytorch/3.3-deferred-init.ipynb) 中所实验的）：

#### 1. 使用 PyTorch 原生 Lazy 模块
PyTorch 引入了 `torch.nn.LazyLinear`、`torch.nn.LazyConv2d` 等模块：
```python
import torch
from torch import nn

# 无需指定 in_features
net = nn.Sequential(
    nn.LazyLinear(256),
    nn.ReLU(),
    nn.LazyLinear(10)
)
print(net[0].weight)  # <UninitializedParameter>，尚未分配内存

# 首次前向传播，触发延后初始化
X = torch.rand(size=(2, 20))
net(X)
print(net[0].weight.shape)  # torch.Size([256, 20])，推断完成并初始化
```

#### 2. 自定义惰性初始化逻辑
在自定义 `nn.Module` 的 `forward` 方法中利用条件判断动态初始化：
```python
class CustomLazyLinear(nn.Module):
    def __init__(self, out_features):
        super().__init__()
        self.out_features = out_features
        self.linear = None

    def forward(self, X):
        if self.linear is None:
            in_features = X.shape[-1]
            self.linear = nn.Linear(in_features, self.out_features)
        return self.linear(X)
```

---

### 四、 课后练习解答与思考

Notebook 末尾提出了 3 个针对初始化与维度的关键思考：

1. **练习 1：如果指定了第一层的输入尺寸，但没有指定后续层的尺寸，会发生什么？是否立即进行初始化？**
   * **分析**：只有完全确定输入/输出尺寸的层可以被立即初始化。对于后续未指定输入尺寸的层，若不传入数据执行前向传播，框架依然无法预先推断出其输入维度（除非框架内部具备静态图类型推导机制提前完成静态 shape 推断）。因此后续层仍然会延迟到首次前向传播时才初始化。

2. **练习 2：如果指定了不匹配的维度会发生什么？**
   * **分析**：在显式定义网络时（例如前一层输出为 64，后一层指定输入为 128），或者延迟推断完成后传入了不同特征维度的输入张量，PyTorch 会在执行矩阵乘法（如 `torch.matmul` 或 `addmm`）时抛出维度不匹配异常（如 `RuntimeError: mat1 and mat2 shapes cannot be multiplied`）。

3. **练习 3：如果输入具有不同的维度，需要做什么？（提示：参数绑定/共享）**
   * **分析**：如果同一个模型需要接受不同维度的输入（例如序列长度可变、或不同尺度的特征），常见方案包括：
     * **自适应池化 / 投影层**：在输入与固定网络之间增加 `nn.AdaptiveAvgPool` 或一个投影线性层，将不同维度的特征投影到统一维度空间。
     * **共享/绑定参数（Parameter Sharing）**：通过共享权重矩阵同时处理多路输入，此时不同分支必须在进入共享层前对齐特征维度。
     * **RNN / Transformer 架构**：天然支持沿序列/时间维度的变长输入，因其权重参数仅与特征隐藏维度绑定，与序列步长无关。

In [2]:
import torch
from torch import nn

In [3]:
class LazyLinear(nn.Module):

    def __init__(self):
        super().__init__()
        self.linear = None
    def forward(self, X):
        if self.linear is None:
            self.linear = nn.Linear(in_features=X.shape[-1], out_features=10)
        return self.linear(X)
        

In [5]:
net = LazyLinear()
X = torch.rand(size= (5,20))
print(net)
print(net(X))
print(net)

LazyLinear()
tensor([[-0.5082, -0.7201,  0.2347,  0.3164, -0.2848,  0.1888,  0.7297, -0.3854,
         -0.1312,  0.1608],
        [-0.6565, -0.4383,  0.5111,  0.0669, -0.4929, -0.0792,  0.4274, -0.2774,
         -0.1528,  0.1921],
        [-0.7808, -0.6577,  0.0442,  0.2941, -0.0820, -0.0050,  0.3071, -0.3565,
          0.1188,  0.1292],
        [-0.4384, -0.6729,  0.2675,  0.4470, -0.4704, -0.1601,  0.5959, -0.4372,
          0.0568, -0.0054],
        [-0.2474, -0.7102,  0.1109,  0.3682, -0.4653, -0.0757,  0.2491, -0.3102,
          0.0091,  0.2136]], grad_fn=<AddmmBackward0>)
LazyLinear(
  (linear): Linear(in_features=20, out_features=10, bias=True)
)


In [6]:
class LazyLinear2(nn.Module):

    def __init__(self):
        super().__init__()
        self.linear = nn.LazyLinear(out_features=10)
    def forward(self, X):
        return self.linear(X)

In [7]:
net = LazyLinear2()
X = torch.rand(size= (5,20))
print(net)
print(net(X))
print(net)

LazyLinear2(
  (linear): LazyLinear(in_features=0, out_features=10, bias=True)
)
tensor([[ 0.2142, -0.5572,  0.3395,  0.1522,  0.2774,  0.1011,  0.0317,  0.2868,
          0.1465,  0.5496],
        [ 0.0512, -0.3016,  0.5072,  0.3484,  0.3175,  0.0451,  0.0767,  0.2834,
         -0.0128,  0.5763],
        [ 0.3856, -0.4611,  0.2795,  0.5073,  0.2474,  0.1258,  0.5723,  0.1622,
          0.0294,  0.6030],
        [ 0.1848, -0.6834,  0.5623,  0.2157,  0.5846,  0.1292,  0.2959,  0.2236,
          0.1529,  0.7325],
        [ 0.4267, -0.5820,  0.3745,  0.1815,  0.1361, -0.0160,  0.4610, -0.0243,
          0.3748,  0.9353]], grad_fn=<AddmmBackward0>)
LazyLinear2(
  (linear): Linear(in_features=20, out_features=10, bias=True)
)


/Users/songchao/WorkSpace/PythonProject/DeepLearn/.venv/lib/python3.9/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
